# Detik Search Scraper (checkpoint CSV per 100 + retry 120s)

Scrape hasil pencarian detik.com (searchnews) lalu ambil artikel full (judul, waktu, isi paragraf).  
Fitur:
- Pagination otomatis (deteksi last page)
- Retry untuk error jaringan (tunggu 120 detik)
- Checkpoint append ke CSV setiap 100 artikel
- Resume jika CSV sudah ada (skip URL yang sudah tersimpan)


In [20]:

import os
import re
import time
import random
import csv
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlencode, urlparse, parse_qs

# ======================
# CONFIG
# ======================
QUERY = "politik indonesia"        # tanpa '+', nanti akan di-encode
SITEID = 3                         # sesuai contoh Anda
RESULT_TYPE = "latest"             # sesuai contoh Anda
START_PAGE = 1
END_PAGE = None                    # None = sampai last page terdeteksi

OUTPUT_CSV = "detik_politik_indonesia_articles.csv"
CHECKPOINT_EVERY = 100

# Retry strategy
MAX_RETRY = 5
WAIT_SECONDS = 120
REQUEST_TIMEOUT = 25

# Optional polite delay (mengurangi risiko rate limit)
SLEEP_BETWEEN_REQUESTS_RANGE = (0.6, 1.6)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
}

BASE_SEARCH_URL = "https://www.detik.com/search/searchnews"

def build_search_url(page: int) -> str:
    params = {
        "query": QUERY,
        "siteid": SITEID,
        "result_type": RESULT_TYPE,
        "page": page,
    }
    return f"{BASE_SEARCH_URL}?{urlencode(params)}"

def polite_sleep():
    lo, hi = SLEEP_BETWEEN_REQUESTS_RANGE
    time.sleep(random.uniform(lo, hi))


In [3]:

# ======================
# NETWORK (retry + wait 120s)
# ======================
def fetch_html(url: str, label: str = "") -> str | None:
    retry = 0
    while retry < MAX_RETRY:
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            return resp.text
        except requests.exceptions.RequestException as e:
            retry += 1
            print(f"[ERROR] {label} | Percobaan {retry}/{MAX_RETRY}\n{e}\nMenunggu {WAIT_SECONDS} detik lalu retry...\n")
            time.sleep(WAIT_SECONDS)
    print(f"[SKIP] Gagal setelah {MAX_RETRY} percobaan: {label} | {url}")
    return None


In [4]:

# ======================
# PARSING (search page)
# ======================
def parse_search_page(html: str) -> tuple[list[str], int | None]:
    """
    Return:
    - list of article URLs (href)
    - detected last page (int) or None if not found
    """
    soup = BeautifulSoup(html, "html.parser")

    # Artikel: ambil href dari <a class="media__link" ...>
    links = []
    for a in soup.find_all("a", class_="media__link"):
        href = a.get("href")
        if href and href.startswith("http"):
            links.append(href)

    # Deduplicate in-order
    seen = set()
    uniq_links = []
    for u in links:
        if u not in seen:
            seen.add(u)
            uniq_links.append(u)

    # Pagination: "last page adalah tag setelah class pagination__range"
    last_page = None
    pag = soup.find("div", class_=re.compile(r"\bpagination\b"))
    if pag:
        range_el = pag.find("a", class_="pagination__range")
        if range_el:
            # cari sibling setelahnya yang berupa <a class="pagination__item">1000</a>
            # BeautifulSoup sibling bisa whitespace, jadi iterasi next_elements
            for nxt in range_el.next_elements:
                if getattr(nxt, "name", None) == "a":
                    cls = nxt.get("class", []) or []
                    if "pagination__item" in cls:
                        txt = (nxt.get_text() or "").strip()
                        if txt.isdigit():
                            last_page = int(txt)
                            break

    # Fallback: ambil max page dari semua link pagination__item yang punya page=
    if last_page is None and pag:
        maxp = None
        for a in pag.find_all("a", class_=re.compile(r"\bpagination__item\b")):
            href = a.get("href") or ""
            if "page=" in href:
                try:
                    q = parse_qs(urlparse(href).query)
                    p = q.get("page", [None])[0]
                    if p and str(p).isdigit():
                        p = int(p)
                        maxp = p if (maxp is None or p > maxp) else maxp
                except Exception:
                    pass
        last_page = maxp

    return uniq_links, last_page


In [5]:

# ======================
# PARSING (article page)
# ======================
def parse_article(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    # Artikel container
    article = soup.find("article", class_=re.compile(r"\bdetail\b"))

    # Judul
    title = None
    h1 = soup.find("h1", class_="detail__title")
    if h1:
        title = h1.get_text(strip=True)

    # Waktu
    published_at = None
    dt = soup.find("div", class_="detail__date")
    if dt:
        published_at = dt.get_text(strip=True)

    # Isi: hanya paragraf di dalam <div class="detail__body-text itp_bodycontent">, exclude noncontent
    content_text = ""
    body = soup.find("div", class_=re.compile(r"\bdetail__body-text\b"))
    if body:
        ps = body.find_all("p")
        parts = []
        for p in ps:
            # skip paragraf yang berada di dalam div.noncontent (Baca juga, dsb.)
            if p.find_parent("div", class_="noncontent") is not None:
                continue
            txt = p.get_text(" ", strip=True)
            if txt:
                parts.append(txt)
        content_text = "\n".join(parts)

    return {
        "url": url,
        "title": title,
        "published_at": published_at,
        "content": content_text,
    }


In [6]:

# ======================
# CHECKPOINT CSV (append every N)
# ======================
def load_existing_urls(csv_path: str) -> set[str]:
    if not os.path.exists(csv_path):
        return set()
    try:
        df = pd.read_csv(csv_path)
        if "url" in df.columns:
            return set(df["url"].dropna().astype(str).tolist())
    except Exception:
        pass
    return set()

def append_rows(csv_path: str, rows: list[dict], header_if_new: bool = True):
    if not rows:
        return

    file_exists = os.path.exists(csv_path)
    fieldnames = ["url", "title", "published_at", "content"]

    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if header_if_new and not file_exists:
            w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k) for k in fieldnames})
        f.flush()
        try:
            os.fsync(f.fileno())
        except Exception:
            pass

existing_urls = load_existing_urls(OUTPUT_CSV)
existing_count = len(existing_urls)
print(f"Resume: {existing_count} URL sudah ada di CSV ({OUTPUT_CSV})")


Resume: 0 URL sudah ada di CSV (detik_politik_indonesia.csv)


In [7]:

# ======================
# MAIN LOOP
# ======================
buffer = []
saved_total = existing_count

# Deteksi last page dari page awal (START_PAGE)
first_url = build_search_url(START_PAGE)
first_html = fetch_html(first_url, label=f"SEARCH page {START_PAGE}")
if first_html is None:
    raise RuntimeError("Gagal mengambil halaman awal search. Coba jalankan ulang.")

first_links, detected_last = parse_search_page(first_html)

if END_PAGE is None:
    END_PAGE = detected_last if detected_last is not None else START_PAGE
print(f"Detected last page: {detected_last} | END_PAGE dipakai: {END_PAGE}")

def process_search_page(page: int, html: str):
    global buffer, saved_total, existing_urls

    links, _ = parse_search_page(html)
    if not links:
        print(f"[WARN] Halaman {page} tidak menemukan link artikel.")
        return

    for i, url in enumerate(links, start=1):
        if url in existing_urls:
            continue

        art_html = fetch_html(url, label=f"ARTICLE page {page} item {i}")
        if art_html is None:
            continue

        row = parse_article(art_html, url)
        # Minimal sanity check
        if not row.get("title") and not row.get("content"):
            # tetap simpan kalau mau, tapi biasanya ini indikasi halaman berbeda / blocked
            print(f"[WARN] Artikel kosong/aneh: {url}")

        buffer.append(row)
        existing_urls.add(url)

        # checkpoint per N
        if len(buffer) >= CHECKPOINT_EVERY:
            append_rows(OUTPUT_CSV, buffer)
            saved_total += len(buffer)
            buffer = []
            print(f"[CHECKPOINT] Tersimpan: {saved_total} (termasuk yang sebelumnya) | Last page processed: {page}")

        polite_sleep()

# Proses START_PAGE sudah diambil
process_search_page(START_PAGE, first_html)

# Lanjut halaman berikutnya
for page in range(START_PAGE + 1, END_PAGE + 1):
    url = build_search_url(page)
    html = fetch_html(url, label=f"SEARCH page {page}")
    if html is None:
        continue
    process_search_page(page, html)
    polite_sleep()

# Flush sisa buffer
if buffer:
    append_rows(OUTPUT_CSV, buffer)
    saved_total += len(buffer)
    buffer = []

print(f"SELESAI. Total URL tersimpan (termasuk resume): {saved_total}. Output: {OUTPUT_CSV}")


Detected last page: 1000 | END_PAGE dipakai: 1000
[WARN] Artikel kosong/aneh: https://20.detik.com/detikupdate/20251208-251208105/video-puan-soal-koalisi-permanen-kita-sedang-berduka-urusan-politik-nanti-saja
[CHECKPOINT] Tersimpan: 100 (termasuk yang sebelumnya) | Last page processed: 10
[CHECKPOINT] Tersimpan: 200 (termasuk yang sebelumnya) | Last page processed: 20
[CHECKPOINT] Tersimpan: 300 (termasuk yang sebelumnya) | Last page processed: 30
[CHECKPOINT] Tersimpan: 400 (termasuk yang sebelumnya) | Last page processed: 40
[CHECKPOINT] Tersimpan: 500 (termasuk yang sebelumnya) | Last page processed: 50
[CHECKPOINT] Tersimpan: 600 (termasuk yang sebelumnya) | Last page processed: 60
[WARN] Artikel kosong/aneh: https://news.detik.com/x/detail/intermeso/20250926/Potret-Harapan-Suara-Rakyat/
[CHECKPOINT] Tersimpan: 700 (termasuk yang sebelumnya) | Last page processed: 70
[CHECKPOINT] Tersimpan: 800 (termasuk yang sebelumnya) | Last page processed: 80
[WARN] Artikel kosong/aneh: https:/

In [ ]:
#coba kita baca
df = pd.read_csv(OUTPUT_CSV)
display(df.head(3))
display(df.tail(3))
# df['content'][9996]

,url,title,published_at,content
9996,https://news.detik.com/pemilu/d-6909357/relawa...,"Relawan Ganjar Gelar Lomba Mural, Ajak Anak Mu...","Sabtu, 02 Sep 2023 13:39 WIB",Tim Koordinasi Relawan Pemenangan Pilpres PDIP...
9997,https://news.detik.com/pemilu/d-6909350/prabow...,Prabowo Tegaskan Tak Akan Impor Energi Bila Ja...,"Sabtu, 02 Sep 2023 13:31 WIB",Ketua Umum Partai Gerindra Prabowo Subianto me...
9998,https://news.detik.com/pemilu/d-6909344/prabow...,Prabowo soal Pogram Jokowi: Yang Benar Kita Te...,"Sabtu, 02 Sep 2023 13:25 WIB",Bakal capres sekaligus Ketum Partai Gerindra P...
